In [1]:
from pathlib import Path
import os
import pickle
import numpy as np

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from google import genai

In [2]:
load_dotenv()

GEMINI_API_KEY=os.getenv('GEMINI_API_KEY')

if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found. Please check your .env file.')


client=genai.Client(api_key=GEMINI_API_KEY)
print('Gemini API connected successfully!')

Gemini API connected successfully!


In [3]:
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded successfully!')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [4]:
DOCS_FOLDER=Path('docs')
DATA_FOLDER=Path('data')

DOCS_FOLDER.mkdir(exist_ok=True)
DATA_FOLDER.mkdir(exist_ok=True)

In [5]:
documents=[]

for file_path in DOCS_FOLDER.glob('*.txt'):
    text=file_path.read_text(encoding='utf-8')
    documents.append({
    'text':text,
    'source':file_path.name
    }
    )
if not documents:
    raise FileNotFoundError('No .txt files found inside the docs folder.')

print(f'loaded {len(documents)} documents.')    

loaded 3 documents.


In [6]:
chunks = []

for document in documents:

    paragraphs = [
        paragraph.strip()
        for paragraph in document["text"].split("\n\n")
        if paragraph.strip()
    ]

    for paragraph in paragraphs:

        chunks.append(
            {
                "text": paragraph,
                "source": document["source"]
            }
        )
if not chunks:
    raise ValueError(
        "No text chunks were created from your documents."
    )

print(f"Created {len(chunks)} chunks.")   

Created 3 chunks.


In [7]:
texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

print("Embeddings created successfully!")

Embeddings created successfully!


In [8]:
for chunk,embedding in zip(chunks,embeddings):
    chunk['embedding']=embedding.tolist()

In [9]:
EMBEDDING_FILE=DATA_FOLDER/'embeddings.pkl'

with open(EMBEDDING_FILE,'wb') as file:
    pickle.dump(chunks,file)

print(f'Embeddings saved successfully: {EMBEDDING_FILE }')   

Embeddings saved successfully: data\embeddings.pkl


In [10]:
with open(EMBEDDING_FILE,'rb') as file:
    saved_chunks=pickle.load(file)

print(f'Loaded {len(saved_chunks)} saved chunks.')    

Loaded 3 saved chunks.


In [11]:
def retrieve_relevent_chunks(question,top_k=3):
    question_embedding=embedding_model.encode(question,normalize_embeddings=True)

    scored_chunks=[]

    for chunk in saved_chunks:

        chunk_embedding=np.array(chunk['embedding'])
        score=np.dot(question_embedding,chunk_embedding)
        score_chunks.append({'score':float(score),'text':chunk['text'],'source':chunk['source']})
    scored_chunks.sort(key=lambda x:x['score'],reverse=True)

    return scored_chunks[:top_k]
        

In [12]:
# import time


# def ask_my_docs(question):

#     retrieved_chunks = retrieve_relevant_chunks(
#         question,
#         top_k=3
#     )

#     useful_chunks = [
#         chunk
#         for chunk in retrieved_chunks
#         if chunk["score"] >= 0.30
#     ]

#     if not useful_chunks:
#         return (
#             "I don't know based on the provided documents.",
#             []
#         )

#     context_parts = []

#     for chunk in useful_chunks:

#         context_parts.append(
#             f"Source: {chunk['source']}\n"
#             f"Content: {chunk['text']}"
#         )

#     context = "\n\n---\n\n".join(context_parts)

#     prompt = f"""
# You are a document-based question answering assistant.

# Answer the user's question using ONLY the information
# provided in the CONTEXT below.

# Do not use outside knowledge.

# If the answer cannot be found in the context,
# say exactly:

# "I don't know based on the provided documents."

# Do not make up information.

# CONTEXT:
# {context}

# QUESTION:
# {question}

# ANSWER:
# """

#     # Try Gemini up to 3 times
#     for attempt in range(3):

#         try:

#             response = client.models.generate_content(
#                 model="gemini-3.8-flash",
#                 contents=prompt
#             )

#             answer = response.text

#             return answer, useful_chunks

#         except Exception as error:

#             error_message = str(error)

#             if "503" in error_message or "UNAVAILABLE" in error_message:

#                 if attempt < 2:

#                     print(
#                         f"Gemini is busy. Retrying... "
#                         f"({attempt + 1}/3)"
#                     )

#                     time.sleep(5)

#                 else:

#                     return (
#                         "Gemini is currently busy. "
#                         "Please try again after a few seconds.",
#                         useful_chunks
#                     )

#             else:

#                 raise error


# print("Ask My Docs function updated successfully!")



# ===========================================================================================================
# aeh code ka use tab hota hai jab time shlot finish ho jata hai to vo error nahi dega msg print karke dega
# ===========================================================================================================


In [12]:
def ask_my_docs(question):

    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_relevant_chunks(
        question,
        top_k=3
    )

    # Step 2: Keep only useful chunks
    useful_chunks = [
        chunk
        for chunk in retrieved_chunks
        if chunk["score"] >= 0.30
    ]

    # Step 3: Create context
    context = "\n\n".join(
        chunk["text"]
        for chunk in useful_chunks
    )

    # Step 4: Create prompt
    prompt = f"""
You are a document question-answering assistant.

Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say "I could not find the answer in the documents."

Answer:
"""

    # Step 5: Send prompt to Gemini
    response = client.interactions.create(
        model="gemini-3.5-flash-lite",
        input=prompt
    )

    # Step 6: Get answer
    answer = response.output_text

    # Step 7: Return answer and sources
    return answer, useful_chunks

In [21]:
# def retrieve_relevant_chunks(question, top_k=3):

#     question_embedding = embedding_model.encode(
#         question,
#         normalize_embeddings=True
#     )

#     scored_chunks = []

#     for chunk in saved_chunks:

#         chunk_embedding = np.array(
#             chunk["embedding"]
#         )

#         score = np.dot(
#             question_embedding,
#             chunk_embedding
#         )

#         scored_chunks.append({
#             "score": float(score),
#             "text": chunk["text"],
#             "source": chunk["source"]
#         })

#     scored_chunks.sort(
#         key=lambda x: x["score"],
#         reverse=True
#     )

#     return scored_chunks[:top_k]


# def ask_my_docs(question):

#     # Step 1: Retrieve relevant chunks
#     retrieved_chunks = retrieve_relevant_chunks(
#         question,
#         top_k=3
#     )

#     # Step 2: Keep only useful chunks
#     useful_chunks = [
#         chunk
#         for chunk in retrieved_chunks
#         if chunk["score"] >= 0.30
#     ]

#     # Step 3: Create context
#     context = "\n\n".join(
#         chunk["text"]
#         for chunk in useful_chunks
#     )

#     # Step 4: Create prompt
#     prompt = f"""
# You are a document question-answering assistant.

# Answer the question using ONLY the provided context.

# Context:
# {context}

# Question:
# {question}

# If the answer is not available in the context,
# say "I could not find the answer in the documents."

# Answer:
# """

#     # Step 5: Send prompt to Gemini
#     response = client.interactions.create(
#         model="gemini-3.5-flash-lite",
#         input=prompt
#     )

#     # Step 6: Get answer
#     answer = response.output_text

#     # Step 7: Return answer and sources
#     return answer, useful_chunks


# # Test
# question = "What is machine learning?"

# answer, sources = ask_my_docs(question)

# print("=" * 60)
# print("ASK MY DOCS")
# print("=" * 60)

# print("\nQuestion:")
# print(question)

# print("\nAnswer:")
# print(answer)

# print("\nSources:")

# for source in sources:
#     print(
#         f"- {source['source']} "
#         f"(similarity: {source['score']:.3f})"
#     )

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain machine learning in two simple sentences."
)

print(response.text)

In [17]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY .env file mein nahi mili!")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain machine learning in one simple sentence."
)

print(response.text)

Machine learning is teaching computers to learn from past data and experience so they can make decisions and solve problems on their own, without being explicitly programmed for every step.


In [18]:
def retrieve_relevant_chunks(question, top_k=3):
    question_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    scores = np.dot(
        embeddings,
        question_embedding
    )

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "score": float(scores[index]),
            "text": saved_chunks[index]["text"],
            "source": saved_chunks[index]["source"]
        })

    return results

In [19]:
# ==========================================
# 7. ASK MY DOCS USING GEMINI
# ==========================================

def ask_my_docs(question):
    results = retrieve_relevant_chunks(
        question,
        top_k=3
    )

    # Keep only sufficiently relevant chunks
    relevant_chunks = [
        item for item in results
        if item["score"] >= 0.30
    ]

    if not relevant_chunks:
        return {
            "answer": "I could not find the answer in the documents.",
            "sources": [],
            "scores": []
        }

    context = "\n\n".join(
        f"Source: {item['source']}\n{item['text']}"
        for item in relevant_chunks
    )

    prompt = f"""
You are an assistant that answers questions about uploaded documents.

Rules:
1. Answer using ONLY the context provided below.
2. Do not make up facts or use outside knowledge.
3. If the context does not contain the answer, say exactly:
   "I could not find the answer in the documents."
4. Give a clear and simple answer.

DOCUMENT CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    answer = response.text

    if not answer:
        answer = "Gemini did not return a text answer."

    return {
        "answer": answer.strip(),
        "sources": list(dict.fromkeys(
            item["source"] for item in relevant_chunks
        )),
        "scores": [
            round(item["score"], 3)
            for item in relevant_chunks
        ]
    }



In [20]:
# ==========================================
# 8. ASK A TEST QUESTION
# ==========================================

question = "What is machine learning?"

result = ask_my_docs(question)

print("\n" + "=" * 50)
print("QUESTION:", question)
print("=" * 50)

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
if result["sources"]:
    for source in result["sources"]:
        print("-", source)
else:
    print("No relevant source found.")

print("\nRELEVANCE SCORES:")
if result["scores"]:
    for score in result["scores"]:
        print("-", score)

print("\nAsk My Docs is ready!")


QUESTION: What is machine learning?

ANSWER:
Machine learning is a branch of artificial intelligence that allows computers to learn patterns from data.

SOURCES:
- ml.txt
- dl.txt

RELEVANCE SCORES:
- 0.788
- 0.464

Ask My Docs is ready!
